# Synthetic Persona Survey: GAAIS Attitudes towards AI

This notebook:
1. Creates **two sets of 100 personas** from survey data (demographics-only vs. demographics + Big Five)
2. Defines **5 GAAIS questions** (Likert scale) drawn from the dataset
3. Queries an LLM via OpenRouter for each persona × question
4. Evaluates responses using the same constraint metrics as the repo (PC1 variance explained, effective dependence)

## Setup

In [ ]:
import os
import time
import random
import numpy as np
import pandas as pd
import requests
from datetime import datetime, timezone
from pathlib import Path
from tqdm.notebook import tqdm

In [ ]:
# --- CONFIGURATION ---
API_KEY = os.getenv("OPENROUTER_API_KEY", "YOUR_KEY_HERE")
MODEL = "openai/gpt-4o-mini"
YEAR = 2024
RUNS = 1
N_PERSONAS = 100

OPENROUTER_API_URL = "https://openrouter.ai/api/v1/chat/completions"

# Paths
DATA_PATH = "SPSS_file_with_raw_data.csv"  # adjust if needed
OUTPUT_DIR = Path("generation/synthetic_data/year_2024")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## 1. Load Survey Data and Create Personas

In [ ]:
# Load raw survey data
df_raw = pd.read_csv(DATA_PATH)

# Sample 100 respondents (or take all if <= 100)
random.seed(42)
if len(df_raw) > N_PERSONAS:
    df_sample = df_raw.sample(n=N_PERSONAS, random_state=42).reset_index(drop=True)
else:
    df_sample = df_raw.copy().reset_index(drop=True)
    N_PERSONAS = len(df_sample)

print(f"Sampled {len(df_sample)} respondents from {len(df_raw)} total")
print(f"Columns available: {list(df_sample.columns[:10])}...")

In [ ]:
# --- Build two persona sets ---

# Education level mapping for natural language
edu_map = {
    "Freshman": "a first-year undergraduate (Freshman)",
    "Sophomore": "a second-year undergraduate (Sophomore)",
    "Junior": "a third-year undergraduate (Junior)",
    "Senior": "a fourth-year undergraduate (Senior)",
}

def big5_label(score):
    """Convert a 1-5 Big Five score to a qualitative label."""
    if score <= 2.0:
        return "very low"
    elif score <= 2.75:
        return "low"
    elif score <= 3.25:
        return "moderate"
    elif score <= 4.0:
        return "high"
    else:
        return "very high"

personas_demo = []   # demographics only
personas_full = []   # demographics + Big Five

for i, row in df_sample.iterrows():
    rid = i + 1
    age = int(row["Age"]) if pd.notna(row["Age"]) else 21
    gender = row["Gender"] if pd.notna(row["Gender"]) else "Person"
    edu = edu_map.get(row["Education"], row["Education"])
    major = row["Major"] if pd.notna(row["Major"]) else "Undeclared"

    # Demographics-only persona
    demo_text = (
        f"I am {age} years old. "
        f"I am a {gender.lower()}. "
        f"I am {edu} at a private university. "
        f"My major is {major}."
    )
    personas_demo.append({"respondent_id": rid, "persona": demo_text})

    # Demographics + Big Five persona
    bfi_o = big5_label(row["BFI_O"]) if pd.notna(row["BFI_O"]) else "moderate"
    bfi_e = big5_label(row["BFI_E"]) if pd.notna(row["BFI_E"]) else "moderate"
    bfi_a = big5_label(row["BFI_A"]) if pd.notna(row["BFI_A"]) else "moderate"
    bfi_n = big5_label(row["BFI_N"]) if pd.notna(row["BFI_N"]) else "moderate"
    bfi_c = big5_label(row["BFI_C"]) if pd.notna(row["BFI_C"]) else "moderate"

    full_text = (
        f"{demo_text} "
        f"On personality traits: my openness is {bfi_o}, "
        f"my extraversion is {bfi_e}, "
        f"my agreeableness is {bfi_a}, "
        f"my neuroticism is {bfi_n}, "
        f"my conscientiousness is {bfi_c}."
    )
    personas_full.append({"respondent_id": rid, "persona": full_text})

df_demo = pd.DataFrame(personas_demo)
df_full = pd.DataFrame(personas_full)

# Save as CSV in repo format
df_demo.to_csv(OUTPUT_DIR.parent.parent / "data" / "personas_demo_only.csv", index=False)
df_full.to_csv(OUTPUT_DIR.parent.parent / "data" / "personas_demo_big5.csv", index=False)

print(f"Created {len(df_demo)} demographics-only personas")
print(f"Created {len(df_full)} demographics+Big5 personas")
print(f"\nExample demographics-only:\n  {df_demo.iloc[0]['persona']}")
print(f"\nExample demographics+Big5:\n  {df_full.iloc[0]['persona']}")

## 2. Define 5 GAAIS Questions

In [ ]:
# 5 items from the GAAIS (Schepman & Rodway, 2020, 2023)
# 3 positive, 2 negative — matching the dataset column names
# Response scale: 1=Strongly disagree, 2=(Somewhat) disagree, 3=Neutral,
#                 4=(Somewhat) agree, 5=Strongly agree

GAAIS_OPTIONS = {
    1: "Strongly disagree",
    2: "(Somewhat) disagree",
    3: "Neutral",
    4: "(Somewhat) agree",
    5: "Strongly agree",
}

QUESTIONS = {
    "GAAIS_5_P": {
        "text": "I am impressed by what Artificial Intelligence can do.",
        "options": GAAIS_OPTIONS,
    },
    "GAAIS_7_P": {
        "text": "Artificial Intelligence can have positive impacts on people's wellbeing.",
        "options": GAAIS_OPTIONS,
    },
    "GAAIS_17_P": {
        "text": "Artificial Intelligence is exciting.",
        "options": GAAIS_OPTIONS,
    },
    "R_GAAIS_3_N": {
        "text": "I find Artificial Intelligence sinister.",
        "options": GAAIS_OPTIONS,
    },
    "R_GAAIS_8_N": {
        "text": "Artificial Intelligence might take control of people.",
        "options": GAAIS_OPTIONS,
    },
}

print(f"Defined {len(QUESTIONS)} GAAIS questions: {list(QUESTIONS.keys())}")

## 3. Query LLM via OpenRouter

In [ ]:
def query_openrouter(model, persona, question, options, api_key, year, max_retries=3):
    """Query OpenRouter for a single persona-question pair."""
    options_text = "\n".join([f"{k}. {v}" for k, v in options.items()])

    prompt = (
        f"It is now {year}. You are answering survey questions as the following person:\n\n"
        f"{persona}\n\n"
        f"Question: {question}\n\nOptions:\n{options_text}\n\n"
        f'Respond with ONLY the number of your answer (e.g., "1" or "2"). Do not explain your reasoning.'
    )

    headers = {"Authorization": f"Bearer {api_key}", "Content-Type": "application/json"}
    body = {
        "model": model,
        "messages": [{"role": "user", "content": prompt}],
        "temperature": 1.0,
        "max_tokens": 50,
    }

    for attempt in range(max_retries):
        try:
            resp = requests.post(OPENROUTER_API_URL, headers=headers, json=body, timeout=30)
            resp.raise_for_status()
            result = resp.json()
            answer_text = result["choices"][0]["message"]["content"].strip()
            usage = result.get("usage", {})

            try:
                answer = int(answer_text)
                if answer not in options:
                    return {"answer": None, "error": f"Invalid: {answer_text}",
                            "prompt_tokens": usage.get("prompt_tokens", 0),
                            "completion_tokens": usage.get("completion_tokens", 0),
                            "raw_response": answer_text}
            except ValueError:
                return {"answer": None, "error": f"Parse error: {answer_text}",
                        "prompt_tokens": usage.get("prompt_tokens", 0),
                        "completion_tokens": usage.get("completion_tokens", 0),
                        "raw_response": answer_text}

            return {"answer": answer, "error": None,
                    "prompt_tokens": usage.get("prompt_tokens", 0),
                    "completion_tokens": usage.get("completion_tokens", 0),
                    "raw_response": answer_text}

        except Exception as e:
            if attempt < max_retries - 1:
                time.sleep(2 ** attempt)
            else:
                return {"answer": None, "error": str(e),
                        "prompt_tokens": 0, "completion_tokens": 0, "raw_response": ""}

In [ ]:
def run_survey(personas_df, condition_label):
    """Run all persona x question queries for one persona set. Returns results DataFrame."""
    tasks = []
    for _, row in personas_df.iterrows():
        for var_name, q in QUESTIONS.items():
            for run in range(1, RUNS + 1):
                tasks.append({
                    "persona_id": row["respondent_id"],
                    "persona_text": row["persona"],
                    "variable": var_name,
                    "question": q["text"],
                    "options": q["options"],
                    "run": run,
                })

    print(f"\n--- {condition_label} ---")
    print(f"Total API calls: {len(tasks)} ({len(personas_df)} personas x {len(QUESTIONS)} questions x {RUNS} runs)")

    results = []
    for task in tqdm(tasks, desc=condition_label):
        r = query_openrouter(
            MODEL, task["persona_text"], task["question"], task["options"], API_KEY, YEAR
        )
        results.append({
            "timestamp": datetime.now(timezone.utc).isoformat(),
            "model": MODEL,
            "condition": condition_label,
            "persona_id": task["persona_id"],
            "variable": task["variable"],
            "question_short": task["question"][:50] + "...",
            "run": task["run"],
            "answer": r.get("answer"),
            "prompt_tokens": r.get("prompt_tokens", 0),
            "completion_tokens": r.get("completion_tokens", 0),
            "total_tokens": r.get("prompt_tokens", 0) + r.get("completion_tokens", 0),
            "error": r.get("error", ""),
            "raw_response": r.get("raw_response", ""),
        })

    return pd.DataFrame(results)

In [ ]:
# Run both conditions
results_demo = run_survey(df_demo, "demo_only")
results_full = run_survey(df_full, "demo_big5")

# Save
model_fn = MODEL.replace("/", "_")
results_demo.to_csv(OUTPUT_DIR / f"{model_fn}_demo_only.csv", index=False)
results_full.to_csv(OUTPUT_DIR / f"{model_fn}_demo_big5.csv", index=False)

print(f"\nSaved {len(results_demo)} demo-only results")
print(f"Saved {len(results_full)} demo+Big5 results")

## 4. Evaluate with Repo Metrics

Replicates the two core constraint metrics from the repo:
- **PC1 variance explained**: share of total variance captured by the first principal component of the correlation matrix
- **Effective dependence (De)**: \\(1 - |R|^{1/k}\\), where \\(|R|\\) is the determinant and \\(k\\) the number of variables

In [ ]:
def pc1_var_explained(R):
    """Share of variance explained by the first eigenvalue (matches repo's pc1_var_explained)."""
    ev = np.linalg.eigvalsh(R)
    ev = np.maximum(ev, 0)
    total = ev.sum()
    if total <= 0:
        return np.nan
    return ev[-1] / total  # eigvalsh returns in ascending order


def effective_dependence(R):
    """Effective dependence De = 1 - |R|^(1/k) (matches repo's effective_dependence)."""
    k = R.shape[0]
    ev = np.linalg.eigvalsh(R)
    if np.any(ev <= 0):
        return 1.0
    gm = np.exp(np.mean(np.log(ev)))
    gm = np.clip(gm, 0, 1)
    return np.clip(1 - gm, 0, 1)


def pivot_to_wide(results_df):
    """Pivot long results to wide format: one row per persona, one column per question."""
    valid = results_df.dropna(subset=["answer"]).copy()
    wide = valid.pivot_table(index="persona_id", columns="variable", values="answer", aggfunc="first")
    return wide.dropna()


def compute_metrics(results_df, label):
    """Compute constraint metrics for one condition."""
    wide = pivot_to_wide(results_df)
    if wide.shape[0] < 5 or wide.shape[1] < 2:
        print(f"  {label}: insufficient data ({wide.shape[0]} rows, {wide.shape[1]} cols)")
        return None

    R = wide.corr(method="spearman").values
    pc1 = pc1_var_explained(R)
    de = effective_dependence(R)

    success = results_df["answer"].notna().sum()
    total = len(results_df)

    return {
        "condition": label,
        "n_personas": wide.shape[0],
        "n_questions": wide.shape[1],
        "success_rate": success / total * 100,
        "pc1_var_explained": pc1,
        "effective_dependence": de,
    }

In [ ]:
# Compute metrics for human baseline from survey data
likert_map = {
    "Strongly disagree": 1, "(Somewhat) disagree": 2, "Neutral": 3,
    "(Somewhat) agree": 4, "Strongly agree": 5,
}
human_cols = list(QUESTIONS.keys())
human_wide = df_sample[human_cols].copy()
for col in human_cols:
    human_wide[col] = human_wide[col].map(likert_map)
human_wide = human_wide.dropna()

R_human = human_wide.corr(method="spearman").values
human_metrics = {
    "condition": "human_survey",
    "n_personas": human_wide.shape[0],
    "n_questions": human_wide.shape[1],
    "success_rate": 100.0,
    "pc1_var_explained": pc1_var_explained(R_human),
    "effective_dependence": effective_dependence(R_human),
}

# Compute metrics for LLM conditions
metrics_demo = compute_metrics(results_demo, "llm_demo_only")
metrics_full = compute_metrics(results_full, "llm_demo_big5")

# Combine and display
all_metrics = [human_metrics]
if metrics_demo:
    all_metrics.append(metrics_demo)
if metrics_full:
    all_metrics.append(metrics_full)

metrics_df = pd.DataFrame(all_metrics)
print("=" * 70)
print("CONSTRAINT METRICS COMPARISON")
print("=" * 70)
print(metrics_df.to_string(index=False))

In [ ]:
# Answer distribution comparison
print("\n" + "=" * 70)
print("ANSWER DISTRIBUTIONS BY QUESTION")
print("=" * 70)

for var in QUESTIONS:
    print(f"\n{var}: {QUESTIONS[var]['text'][:60]}...")

    # Human
    h_vals = df_sample[var].map(likert_map).dropna()
    h_dist = h_vals.value_counts().sort_index()
    print(f"  Human (n={len(h_vals)}):     {h_dist.to_dict()}")

    # Demo-only LLM
    d_vals = results_demo[results_demo["variable"] == var]["answer"].dropna()
    d_dist = d_vals.value_counts().sort_index()
    print(f"  LLM demo-only (n={len(d_vals)}): {d_dist.to_dict()}")

    # Demo+Big5 LLM
    f_vals = results_full[results_full["variable"] == var]["answer"].dropna()
    f_dist = f_vals.value_counts().sort_index()
    print(f"  LLM demo+B5 (n={len(f_vals)}):  {f_dist.to_dict()}")